# Capital Raise Detector

Predict the likelihood of companies performing primary equity raises based on 5 financial signals.

## Overview

This notebook analyzes companies on:
1. **Cash Runway** - How many months of cash remains
2. **Liquidity Stress** - Current/quick ratios and working capital
3. **Debt Maturity** - Near-term debt obligations
4. **Operational Red Flags** - Revenue trends, margins, FCF
5. **Market & Behavioral Signals** - Stock price, insider activity

Companies scoring > 50 are flagged for potential capital raise activity.

In [28]:
# Import required modules
from analyzer import CapitalRaiseAnalyzer
from data import EXAMPLE_COMPANIES, EARNINGS_CALL_TRANSCRIPT, MARKET_NEWS
from models import FinancialMetrics
import pandas as pd
from IPython.display import display, Markdown


## Initialize Analyzer

Create analyzer instance with your preferred LLM model.

In [29]:
# Initialize with default model (gpt-4o-mini)
# For higher accuracy, use 'gpt-4' or 'gpt-4o'
analyzer = CapitalRaiseAnalyzer(model_name="gpt-4o-mini")
print("✓ Analyzer initialized")

✓ Analyzer initialized


## Analyze Example Companies

Run the detector on our 5 example companies with different risk profiles.

In [30]:
# Analyze all example companies
predictions = analyzer.batch_analyze(EXAMPLE_COMPANIES)

# Display summary
summary_data = [
    {
        "Ticker": p.ticker,
        "Company": p.company_name,
        "Score": f"{p.likelihood_score:.1f}",
        "Risk Level": p.risk_level.upper(),
        "Alert": "⚠️  YES" if p.above_threshold else "✓ No",
        "Confidence": f"{p.confidence:.0f}%",
    }
    for p in predictions
]

summary_df = pd.DataFrame(summary_data)
display(Markdown("### Summary of All Companies"))
display(summary_df)

### Summary of All Companies

,Ticker,Company,Score,Risk Level,Alert,Confidence
0,STRONG,Strong Tech Inc,0.0,LOW,✓ No,80%
1,BURN,Burn Fast Corp,74.0,HIGH,⚠️ YES,85%
2,STRESS,Struggling Industries,98.5,CRITICAL,⚠️ YES,85%
3,GROW,Growth Ventures LLC,39.0,LOW,✓ No,80%
4,STABLE,Mature Corp Inc,0.0,LOW,✓ No,80%


## Detailed Analysis by Company

Inspect individual company scores and key drivers.

In [31]:
# Show detailed results for each company
for pred in predictions:
    print(pred)


Capital Raise Detector Report
Company: Strong Tech Inc (STRONG)
Sector: Information Technology | Market Cap: $15.0B
Likelihood Score: 0.0/100
Risk Level: LOW
Status: ✓ Below threshold
Confidence: 80.0%

Signal Breakdown:
  • Cash Runway: 0.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 0.0/15
  • Operational Red Flags: 0.0/15
  • Market & Behavioral: 0.0/10

Key Drivers:



Capital Raise Detector Report
Company: Burn Fast Corp (BURN)
Sector: Energy | Market Cap: $1.8B
Likelihood Score: 74.0/100
Risk Level: HIGH
Status: ⚠️  ABOVE THRESHOLD
Confidence: 85.0%

Signal Breakdown:
  • Cash Runway: 25.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 18.0/15
  • Operational Red Flags: 15.0/15
  • Market & Behavioral: 16.0/10

Key Drivers:
  - Low cash runway: 5.0 months
  - $20M debt due in 12 months
  - Negative operating cash flow
  - Elevated insider selling activity
  - Stock down 62% from 52-week high


Capital Raise Detector Report
Company: Struggling Industries (STRESS)
Secto

## Signal Breakdown

Visualize the contribution of each signal to the overall score.

In [32]:
# Create signal breakdown for each company
signal_data = []
for p in predictions:
    signal_data.append({
        "Ticker": p.ticker,
        "Cash Runway (0-40)": f"{p.signal_scores.cash_runway_score:.1f}",
        "Liquidity (0-20)": f"{p.signal_scores.liquidity_stress_score:.1f}",
        "Debt Maturity (0-15)": f"{p.signal_scores.debt_maturity_score:.1f}",
        "Operational (0-15)": f"{p.signal_scores.operational_red_flags_score:.1f}",
        "Market (0-10)": f"{p.signal_scores.market_behavioral_score:.1f}",
        "Total (0-100)": f"{p.likelihood_score:.1f}",
    })

signals_df = pd.DataFrame(signal_data)
display(Markdown("### Signal Score Breakdown"))
display(signals_df)

### Signal Score Breakdown

,Ticker,Cash Runway (0-40),Liquidity (0-20),Debt Maturity (0-15),Operational (0-15),Market (0-10),Total (0-100)
0,STRONG,0.0,0.0,0.0,0.0,0.0,0.0
1,BURN,25.0,0.0,18.0,15.0,16.0,74.0
2,STRESS,25.0,8.5,25.0,20.0,20.0,98.5
3,GROW,25.0,0.0,1.0,11.0,2.0,39.0
4,STABLE,0.0,0.0,0.0,0.0,0.0,0.0


## Alerts

Companies scoring above 50 (threshold) that need monitoring.

In [33]:
# Get companies above threshold
alerts = analyzer.get_alerts(predictions)

if alerts:
    display(Markdown(f"### ⚠️  {len(alerts)} Companies Above Threshold"))
    for alert in alerts:
        print(f"\n**{alert.ticker}** - Score: {alert.likelihood_score:.1f}")
        print(f"Risk Level: {alert.risk_level.upper()}")
        print("Key Drivers:")
        for driver in alert.key_drivers:
            print(f"  • {driver}")
else:
    print("No companies above threshold.")

### ⚠️  2 Companies Above Threshold


**STRESS** - Score: 98.5
Risk Level: CRITICAL
Key Drivers:
  • Low cash runway: 3.8 months
  • Weak liquidity: Current ratio 0.82
  • $50M debt due in 12 months
  • Negative operating cash flow
  • Revenue decline: -16.7%

**BURN** - Score: 74.0
Risk Level: HIGH
Key Drivers:
  • Low cash runway: 5.0 months
  • $20M debt due in 12 months
  • Negative operating cash flow
  • Elevated insider selling activity
  • Stock down 62% from 52-week high


## With Market Signal Analysis

Include LLM analysis of earnings calls and news (requires OpenAI API key).

In [34]:
# Re-analyze with market signals
# This uses the LLM to analyze earnings call transcripts and news
# WARNING: This makes API calls to OpenAI

print("Analyzing company with market signals...")
enhanced_pred = analyzer.analyze(
    EXAMPLE_COMPANIES[1],  # BURN company
    earnings_call_transcript=EARNINGS_CALL_TRANSCRIPT,
    market_news=MARKET_NEWS,
)

print(enhanced_pred)
print(f"\nMarket signals added: {len(enhanced_pred.key_drivers)} key drivers identified")

Analyzing company with market signals...

Capital Raise Detector Report
Company: Burn Fast Corp (BURN)
Sector: Energy | Market Cap: $1.8B
Likelihood Score: 74.0/100
Risk Level: HIGH
Status: ⚠️  ABOVE THRESHOLD
Confidence: 85.0%

Signal Breakdown:
  • Cash Runway: 25.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 18.0/15
  • Operational Red Flags: 15.0/15
  • Market & Behavioral: 16.0/10

Key Drivers:
  - Low cash runway: 5.0 months
  - $20M debt due in 12 months
  - Negative operating cash flow
  - Elevated insider selling activity
  - Stock down 62% from 52-week high


Market signals added: 5 key drivers identified


## Custom Company Analysis

Analyze your own company data.

In [35]:
# Example: Create custom company
from models import FinancialMetrics

my_company = FinancialMetrics(
    ticker="MYCORP",
    company_name="My Custom Company",
    cash_and_equivalents=100_000_000,
    monthly_burn_rate=5_000_000,
    current_assets=150_000_000,
    current_liabilities=80_000_000,
    quick_assets=120_000_000,
    accounts_payable=30_000_000,
    total_debt=50_000_000,
    debt_due_12mo=10_000_000,
    debt_due_6_18mo=15_000_000,
    revenue_trailing_12m=500_000_000,
    revenue_prior_year=450_000_000,
    operating_cash_flow_trailing_12m=50_000_000,
    gross_margin=0.60,
    prior_gross_margin=0.58,
    capex_annual=20_000_000,
    stock_price=100.0,
    stock_price_52w_high=110.0,
    insider_selling_activity="normal",
    auditor_going_concern=False,
    credit_rating="BBB",
    credit_rating_outlook="stable",
)

# Analyze
custom_result = analyzer.analyze(my_company)
print(custom_result)


Capital Raise Detector Report
Company: My Custom Company (MYCORP)
Sector: Unknown | Market Cap: Unknown
Likelihood Score: 0.0/100
Risk Level: LOW
Status: ✓ Below threshold
Confidence: 80.0%

Signal Breakdown:
  • Cash Runway: 0.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 0.0/15
  • Operational Red Flags: 0.0/15
  • Market & Behavioral: 0.0/10

Key Drivers:




## Analyze by Ticker (Auto-fetch from SEC)

Enter any US public company ticker and automatically pull financial metrics from SEC EDGAR filings and Yahoo Finance.

In [36]:
from sec_fetcher import fetch_metrics

# Change this ticker to any US public company
TICKER = "GKOS"

# Fetch metrics automatically from SEC EDGAR + Yahoo Finance
metrics = fetch_metrics(TICKER)

# Run the analysis
result = analyzer.analyze(metrics)
print(result)

Looking up GKOS...
  Found: GLAUKOS Corp (CIK: 1192448)
  Fetching SEC filings...
  Fetching credit rating...
  Fetching insider activity (Form 4)...
  Checking for going concern warnings...
  Fetching stock data...
  Done! Metrics loaded for GLAUKOS Corp

Capital Raise Detector Report
Company: GLAUKOS Corp (GKOS)
Sector: Healthcare | Market Cap: $6.3B
Likelihood Score: 17.0/100
Risk Level: LOW
Status: ✓ Below threshold
Confidence: 80.0%

Signal Breakdown:
  • Cash Runway: 0.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 0.0/15
  • Operational Red Flags: 15.0/15
  • Market & Behavioral: 2.0/10

Key Drivers:
  - Negative operating cash flow



## Screen All US Public Companies

Scan all ~10,000+ SEC filers and find companies flagged as high risk for capital raises. Uses the SEC EDGAR XBRL Frames API for efficient bulk data retrieval.

**Note:** This takes 1-2 minutes to fetch data from SEC EDGAR.

In [37]:
# SETUP: Ensure yfinance is installed for sector data fetching
import subprocess
import sys

print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

try:
    import yfinance
    print("✓ yfinance is already installed")
    # Test it works
    import yfinance as yf
    test = yf.Ticker("AAPL")
    sector = test.info.get("sector")
    print(f"✓ yfinance test successful - AAPL sector: {sector}")
except ImportError as e:
    print(f"⚠️ yfinance not found. Installing...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
        import yfinance as yf
        test = yf.Ticker("AAPL")
        sector = test.info.get("sector")
        print(f"✓ yfinance installed and working - AAPL sector: {sector}")
    except Exception as install_error:
        print(f"✗ Failed to install yfinance: {install_error}")
except Exception as e:
    print(f"✗ yfinance error: {type(e).__name__}: {e}")

Python version: 3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 (clang-1600.0.26.6)]
Python executable: /Users/anishakarande/Documents/GitHub/ai-class-project/Untitled/.venv/bin/python
✓ yfinance is already installed
✓ yfinance test successful - AAPL sector: Technology


In [40]:
import importlib
from screener import screen_all_companies

# reload module in case it was changed earlier
importlib.reload(__import__('screener'))

# Screen all US public companies
# Uses SEC EDGAR Frames API (bulk data, ~13 API calls total)
# Show companies with score >= 40 (includes below-threshold companies)
all_risk = screen_all_companies(min_score=40)

# Display results as a table
if all_risk:
    # Build table with 5 signal scores as columns
    screen_data = []
    for ticker, name, pred in all_risk[:50]:  # Show top 50
        row = {
            "Ticker": ticker,
            "Company": name,
            "Sector": pred.sector,
            "Market Cap": f"${pred.market_cap/1e9:.1f}B" if pred.market_cap > 0 else "Unknown",
            "Cash Runway (0-40)": f"{pred.signal_scores.cash_runway_score:.1f}",
            "Liquidity (0-20)": f"{pred.signal_scores.liquidity_stress_score:.1f}",
            "Debt Maturity (0-15)": f"{pred.signal_scores.debt_maturity_score:.1f}",
            "Operational (0-15)": f"{pred.signal_scores.operational_red_flags_score:.1f}",
            "Market (0-10)": f"{pred.signal_scores.market_behavioral_score:.1f}",
            "Total Score": f"{pred.likelihood_score:.1f}",
            "Risk Level": pred.risk_level.upper(),
        }
        screen_data.append(row)

    screen_df = pd.DataFrame(screen_data)
    display(Markdown(f"### Companies with Capital Raise Risk ({len(all_risk)} total scoring 40+, showing top 50)"))
    display(Markdown("**Score > 50 = Above threshold (alert) | Score 40-50 = Below threshold (watch list)**"))
    display(screen_df)
else:
    print("No companies found.")

Screening NYSE/Nasdaq companies (CY2025 Q1-Q4)...
  Minimum market cap: $1000M
Step 1: Fetching financial data from SEC EDGAR...
  Fetching balance sheet data (Q1-Q4)...
    Q1 balance sheet done (8 calls)
    Q2 balance sheet done (16 calls)
    Q3 balance sheet done (24 calls)
    Q4 balance sheet done (32 calls)
  Fetching revenue data (Q1-Q4)...
  Fetching cash flow data...
  Fetching profitability data...
  Fetching public float data (Q1-Q4)...
  Total API calls: 61
  Fetching prior year revenue (Q1-Q4)...
  Fetching prior year gross profit (Q1-Q4)...
Step 2: Building company list...
  5823 companies on NYSE/Nasdaq
  4259 with financial data
Step 3: Scoring companies...
    GPC: Consumer Discretionary
    MMS: Information Technology
    SDHC: Industrials
    WAL-PA: Financials
    WHR: Information Technology
    FUN: Consumer Discretionary
    MARA: Financials
    CLSKW: Financials
    AVAV: Industrials
    KBH: Industrials
    AVNT: Materials
    FNKO: Industrials
    GOLF: Indus

### Companies with Capital Raise Risk (91 total scoring 40+, showing top 50)

**Score > 50 = Above threshold (alert) | Score 40-50 = Below threshold (watch list)**

,Ticker,Company,Sector,Market Cap,Cash Runway (0-40),Liquidity (0-20),Debt Maturity (0-15),Operational (0-15),Market (0-10),Total Score,Risk Level
0,PLCE,"Childrens Place, Inc.",Consumer Discretionary,$0.4B,25.0,8.5,15.0,19.0,0.0,67.5,HIGH
1,HCXY,"Hercules Capital, Inc.",Unknown,$3.3B,25.0,10.0,15.0,16.0,0.0,66.0,HIGH
2,CBRL,"CRACKER BARREL OLD COUNTRY STORE, INC",Consumer Discretionary,$1.2B,25.0,10.0,15.0,11.0,0.0,61.0,HIGH
3,CLF,CLEVELAND-CLIFFS INC.,Energy,$3.7B,25.0,1.0,15.0,19.0,0.0,60.0,HIGH
4,BALL,BALL Corp,Industrials,$15.3B,15.0,3.5,20.0,20.0,0.0,58.5,MEDIUM
5,SPWH,"SPORTSMAN'S WAREHOUSE HOLDINGS, INC.",Consumer Discretionary,$0.4B,25.0,2.5,20.0,11.0,0.0,58.5,MEDIUM
6,SPH,SUBURBAN PROPANE PARTNERS LP,Consumer Discretionary,$1.4B,25.0,2.5,20.0,11.0,0.0,58.5,MEDIUM
7,FUN,Six Flags Entertainment Corporation/NEW,Consumer Discretionary,$3.1B,25.0,10.0,15.0,8.0,0.0,58.0,MEDIUM
8,HURN,Huron Consulting Group Inc.,Information Technology,$2.3B,25.0,0.0,20.0,11.0,0.0,56.0,MEDIUM
9,VSECU,VSE CORP,Information Technology,$2.6B,25.0,0.0,20.0,11.0,0.0,56.0,MEDIUM


### Why Multiple Companies Show Score ~51?

In bulk screening, companies often score similarly (~50-51) because:

1. **Limited Input Data**: EDGAR XBRL data is standardized but incomplete
   - No stock price data (accounts for 0-10 points from market signals)
   - No insider activity data (all set to "normal")
   - Credit rating data often unavailable
   
2. **Uniform Debt Estimation**: Without detailed debt maturity schedules, companies with similar debt levels get similar scores

3. **Solution**: For differentiation, analyze companies individually using:
   - **Custom Company Analysis** section above (add real market data)
   - **Auto-fetch from SEC** section (adds stock price + insider data from Yahoo Finance)
   - These provide full context including market signals (0-10 additional points)

In [42]:
# Simple backtest demo with a synthetic raise list
# (in practice you'd load actual event tickers from EDGAR filings)

# score 2020 universe
results_2020 = screen_all_companies(year=2020, min_score=0)
# pretend companies that scored >50 in 2020 went on to raise in 2021
synthetic_raises = [t for t,_,p in results_2020 if p.likelihood_score > 50]

# evaluate
bt_df = pd.DataFrame([
    {"ticker": t, "score": p.likelihood_score, "raised_next": t in synthetic_raises}
    for t,_,p in results_2020
])

print("Number of companies evaluated:", len(bt_df))
print("Number of synthetic raises:", len(synthetic_raises))
print("Fraction of synthetically-raised companies scored >50:",
      bt_df[bt_df.raised_next].score.gt(50).mean())

# show top 20 by score
display(Markdown("### Top 20 Companies by Risk Score (2020)"))
display(bt_df.nlargest(20, 'score'))

Screening NYSE/Nasdaq companies (CY2020 Q1-Q4)...
  Minimum market cap: $1000M
Step 1: Fetching financial data from SEC EDGAR...
  Fetching balance sheet data (Q1-Q4)...
    Q1 balance sheet done (8 calls)
    Q2 balance sheet done (16 calls)
    Q3 balance sheet done (24 calls)
    Q4 balance sheet done (32 calls)
  Fetching revenue data (Q1-Q4)...
  Fetching cash flow data...
  Fetching profitability data...
  Fetching public float data (Q1-Q4)...
  Total API calls: 61
  Fetching prior year revenue (Q1-Q4)...
  Fetching prior year gross profit (Q1-Q4)...
Step 2: Building company list...
  5823 companies on NYSE/Nasdaq
  3720 with financial data
Step 3: Scoring companies...
    MHK: Materials
    BRX: Real Estate
    IART: Industrials
    SREA: Utilities
    ONTO: Industrials
    MDB: Information Technology
    GPC: Consumer Discretionary
    MMS: Information Technology
    PLMR: Financials
    SDGR: Materials
    WY: Real Estate
    NWS: Materials
    HLNE: Financials
    HBANZ: Fina

### Top 20 Companies by Risk Score (2020)

,ticker,score,raised_next
0,FE,67.0,True
1,NIOBW,66.0,True
2,LESL,63.0,True
3,BOKF,61.0,True
4,CVNA,59.5,True
5,MSGS,59.0,True
6,LTH,58.0,True
7,TXT,56.0,True
8,AXP,55.0,True
9,PFSI,54.0,True



### Backtest Example (Avoiding Look‑Ahead Bias)

To simulate a realistic deployment, compute scores using only the financial data that was
available *before* a given cut‑off date, then compare against actual raise events that
occurred afterward. You can do this by passing the desired year to `screen_all_companies`
(or using `fetch_metrics` directly) and then checking whether each ticker subsequently
filed a financing document in the next period.

```python
# 1. define your event list (e.g. parse EDGAR for S-1/S-3 filings in 2021)
raises_2021 = load_raise_events("data/raises_2021.csv")  # returns list of tickers

# 2. score all companies using only 2020 numbers
results_2020 = screen_all_companies(year=2020, min_score=0)

# 3. label the universe
df = pd.DataFrame([
    {"ticker": t, "score": p.likelihood_score, "raised_next_year": t in raises_2021}
    for t,_,p in results_2020
])

# 4. evaluate performance
print(df[df.raised_next_year].score.describe())      # scores for companies that actually raised
print(df[df.score>50].raised_next_year.mean())       # proportion flagged that raised
```

Adjust the `year` (or add quarter logic) to shift the window; this ensures you never
feed the model any data filed after the raise event, avoiding look‑ahead bias.

## Score Interpretation Guide

### Risk Levels
- **Critical** (75-100): Immediate capital raise very likely
- **High** (60-74): Strong likelihood of capital raise in next 12 months
- **Medium** (40-59): Moderate risk, monitor closely
- **Low** (0-39): No immediate capital raise pressure

### Threshold
- **Score > 50**: Company flagged for monitoring
- **Score ≤ 50**: Below concern threshold

### Key Signals
1. **Cash Runway** (0-40 pts): Most important. < 12 months is concerning.
2. **Liquidity Stress** (0-20 pts): Current ratio < 1.0 is critical.
3. **Debt Maturity** (0-15 pts): Large obligations due in 6-18 months.
4. **Operational Red Flags** (0-15 pts): Revenue decline, negative FCF.
5. **Market & Behavioral** (0-10 pts): Stock down, insider selling.
